[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/curvature/curvature.ipynb)

# Gravitational Wave Curvature & Tidal Forces in Spacetime Algebra

A passing gravitational wave stretches a ring of freely falling beads one way and squeezes it the other. The curvature behind that tide is a map on the planes of spacetime, `Bivector <- Bivector`, built here from two dyads of the wave's direction. For a plane wave in vacuum it is nonzero, yet applied twice it gives zero: all six of its eigenvalues vanish. Binding an observer's velocity turns it into the tidal map, `Vector <- Vector`, whose eigenvalues are the stretch and squeeze that observer measures. Batching over time and polarization, and broadcasting over the beads, gives the whole detector response in a few lines.

In general relativity the curvature reads as the Riemann tensor $R_{abcd}$, a linear map on the six-dimensional space of bivectors.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from numga import NumpyContext, stack
from numga.algebras import STA
from examples.animation import save_animation
from examples.relativity.curvature import render
from examples.relativity.curvature.core import wave_packet, integrate_acceleration

# The spacetime algebra, with signature (+, -, -, -).
ctx = NumpyContext(STA)
mv = ctx.multivector

Scalar = STA.gatype.scalar()
# Events, velocities and separations are vectors; oriented areas of spacetime are bivectors.
Vector = STA.gatype.vector()
Bivector = STA.gatype.bivector()
# Rotations and boosts.
Rotor = STA.gatype.rotor()

# The curvature, from an area to a plane; the tidal map, from a separation to a relative
# acceleration; the strain, from a rest separation to a displacement; and bilinear forms.
Curvature = STA.gatype((Bivector, Bivector))  # Bivector <- Bivector
Tidal = STA.gatype((Vector, Vector))          # Vector <- Vector
Strain = STA.gatype((Vector, Vector))         # Vector <- Vector
Form = STA.gatype((Scalar, Vector, Vector))   # Scalar <- (Vector, Vector)

t, x, y, z = mv.vector(np.eye(4))
I = mv.txyz

np.set_printoptions(precision=6, suppress=True)

## 1. Curvature from Null Dyads

A plane wave travels along the lightlike direction `k = t + z`. The planes `k.wedge(x)` and `k.wedge(y)` are null: each has zero magnitude, and they are orthogonal to each other. A dyad `n * (n | Bivector)` sends any plane to `n`, scaled by its overlap with `n`, so a sum of such dyads with the plane left open is a curvature. Two dyads with opposite weights make the Ricci contraction vanish: curvature in vacuum. Because the outputs are null and orthogonal to each other, the map sends its own image to zero: all six eigenvalues vanish, yet the map does not.

In [ ]:
# 1. The wave's lightlike direction, k | k == 0, and the two null planes it spans with the transverse axes:
k = t + z                                                          # [] Vector
nx, ny = k.wedge(x), k.wedge(y)                                    # [] Bivector

# 2. Curvature as a sum of dyads. `nx | Bivector` leaves the area open: a map Scalar <- Bivector,
#    the overlap of any area with nx. Multiplying nx by that overlap gives a map Bivector <- Bivector
#    that sends every area onto nx, scaled by how much of nx it contains. Opposite weights on the
#    two dyads cancel the Ricci contraction: vacuum curvature.
plus: Curvature = nx * (nx | Bivector) - ny * (ny | Bivector)      # [] Bivector <- Bivector

# 3. The cross polarization is the plus pattern turned by 45 degrees about the wave axis. For a null
#    curvature that turn is a duality rotation: the pseudoscalar I squares to minus one and commutes with
#    every bivector, `(I * theta).exp()` turns the pattern by half of theta about the axis, the mark of a
#    spin-two field, and a quarter duality turn, -I, is the eighth turn:
cross: Curvature = -I * plus                                       # [] Bivector <- Bivector

# 4. Nonzero, yet nilpotent: the image of plus consists of null planes containing k, which plus annihilates:
singular_values: Scalar = plus.svdvals()                           # [n_singular] Scalar
# Calling a map on a map composes them: plus(plus) applies plus twice, and is the zero map:
singular_values_squared: Scalar = plus(plus).svdvals()             # [n_singular] Scalar
eigenvalues: Scalar = plus.eigvals()                               # [n_eigen] Scalar

print("singular values of plus       :", singular_values.to_array())
print("singular values of plus(plus) :", singular_values_squared.to_array())
print("eigenvalues of plus           :", eigenvalues.to_array().real)

## 2. Symmetries of the Curvature

Three properties of the curvature are statements about one map. Pair symmetry: the curvature is self-adjoint under the inner product of planes. The first Bianchi identity: a cyclic sum of contractions, which with the third vector left open is a map that must vanish. Vacuum: the Ricci form vanishes; with the observer and the separation left open, the contraction is a trace of the curvature against its wedge slot. No frame and no reciprocal basis are needed.

In index notation the three read as $R_{abcd} = R_{cdab}$, $R_{a[bcd]} = 0$ and $R_{bd} = R^a{}_{bad} = 0$.

In [ ]:
# Random spacetime vectors and two area elements built from them:
a, b, c, d = mv.vector(np.random.default_rng(7).normal(size=(4, 4)))       # [] Vector each
ab, cd = a.wedge(b), c.wedge(d)                                             # [] Bivector

# 1. Pair symmetry: the curvature is self-adjoint under the bivector inner product:
left: Scalar = ab | plus(cd)                                                # [] Scalar
right: Scalar = plus(ab) | cd                                               # [] Scalar
pair: Scalar = stack((left, right))                                         # [n_sides] Scalar

# 2. First Bianchi identity, with the third vector left open: the cyclic sum is the zero map:
bianchi = plus(ab).commutator(Vector) + plus(b.wedge(Vector)).commutator(a) + plus(Vector.wedge(a)).commutator(b)  # [] Vector <- Vector
bianchi_singular_values: Scalar = bianchi.svdvals()                         # [n_singular] Scalar

# 3. Vacuum: the Ricci form, with observer and separation open. Vector.commutator(...) is the inner
#    product with an open vector; tracing the output against the wedge slot contracts the rest.
#    Two bare `Vector`s leave two slots open, so ricci is a form: it takes two vectors and returns
#    a scalar. Calling it with both, ricci(a, b), evaluates it:
ricci: Form = Vector.commutator(plus(Vector.wedge(Vector))).trace(slot=1)   # [] Scalar <- (Vector, Vector)
# Zero on every pair:
ricci_on_samples: Scalar = stack((ricci(a, b), ricci(c, d), ricci(a, a)))   # [n_samples] Scalar

print("pair symmetry          :", pair.to_array())
print("Bianchi singular values:", bianchi_singular_values.to_array())
print("Ricci on sample vectors:", ricci_on_samples.to_array())
print(ricci)

## 3. The Observer's Tidal Map

Fix an observer with velocity `t`. `t.wedge(Vector)` sweeps a neighbouring bead's separation into a small spacetime ribbon, the curvature maps the ribbon to a plane, and `.commutator(t)` lets that plane act on the observer again. With the separation left open this is the tidal map, from separation to relative acceleration. It composes different maps rather than conjugating one, so its eigenvalues need not be those of the curvature: the amplitude along x, minus the amplitude along y, and zero along the wave and along time.

In index notation this reads as the geodesic deviation equation, $\frac{D^2 \xi^a}{d\tau^2} = -R^a{}_{bcd}\, u^b \xi^c u^d$, with $u$ the observer and $\xi$ the open separation; its overall sign depends on the curvature convention.

In [ ]:
# 1. Bind the observer twice and leave the separation open (Vector <- Vector):
#    t.wedge(Vector) sweeps a separation into a ribbon, plus(...) maps the ribbon to a null plane,
#    and .commutator(t) contracts that plane with the observer into a relative acceleration.
tidal_plus: Tidal = plus(t.wedge(Vector)).commutator(t)              # [] Vector <- Vector
tidal_eigenvalues: Scalar = tidal_plus.eigvals()                     # [n_eigen] Scalar

print("tidal eigenvalues:", tidal_eigenvalues.to_array().real)

# 2. The same composition, drawn: the ribbon t ^ x, its image (a null plane), and that image's image (zero):
fig = render.draw_curvature_map(plus, observer=t, wave=k, edge=x)
plt.show()

## 4. Boosted Observers: the Same Wave, Doppler Shifted

An observer chasing the wave sees it redshifted; one flying into it sees it blueshifted. The curvature has two time slots, so the tidal amplitude scales with the frequency squared: by the exponential of minus twice the rapidity along the wave. Boost rotors give a whole batch of observers at once, and binding them to the curvature gives a batch of tidal maps whose singular values are that amplitude.

In the notation of special relativity, with rapidity $\zeta$, the amplitude reads as $A\, e^{-2\zeta}$.

In [ ]:
# 1. Observers boosted along the wave direction, batched over rapidity:
rapidities = np.linspace(-0.7, 0.7, 15)
boosts: Rotor = (mv.zt * (rapidities / 2)).exp()                       # [n_obs] Rotor
observers: Vector = boosts >> t                                         # [n_obs] Vector

# 2. Bind each observer to the same curvature; the batch axis carries through the composition. The
#    amplitude is the largest singular value:
responses: Tidal = plus(observers.wedge(Vector)).commutator(observers)  # [n_obs] Vector <- Vector
amplitudes: Scalar = responses.svdvals()[..., 0]                        # [n_obs] Scalar

fig = render.draw_doppler(rapidities, amplitudes)
plt.show()

## 5. A Wave Packet in Three Polarizations

For a weak wave the curvature is minus half the second time derivative of the strain. The strain profile is a phasor, a scalar plus a pseudoscalar: a Gaussian envelope over a few carrier cycles, times the carrier `(I * -phase).exp()`. Times plus, its scalar part alone is the plus wave and its pseudoscalar part alone the cross wave, a quarter cycle behind, since cross is `-I * plus`; the whole phasor times plus is the circular wave. Stacking the three gives one curvature batch over time and polarization.

In index notation the weak-wave curvature reads as $R_{0i0j} = -\tfrac{1}{2}\ddot h_{ij}$, with a Gaussian envelope.

In [ ]:
# 1. The strain profile and its second derivative, as phasors over time:
time = np.linspace(0.0, 6.0, 1201)
strain, second = wave_packet(time, duration=6.0, cycles=9, amplitude=1e-4)   # [n_time] Phasor each

# 2. Multiplying a map by a batch of phasors gives a batch of maps, one per time step; stacking
#    three such batches adds a polarization axis. The scalar part alone, the pseudoscalar part alone,
#    and the whole phasor give the plus, cross and circular waves over (time, polarization):
circular = second * plus                                                     # [n_time] Bivector <- Bivector
waves: Curvature = -0.5 * stack((second.select[0] * plus, second.select[4] * plus, circular), axis=1)  # [n_time, n_polarizations] Bivector <- Bivector

fig = render.draw_packet(time, strain, second)
plt.show()


## 6. A Ring of Freely Falling Beads

Bind the observer to the whole batch and the tidal map follows along: `[n_time, 3] Vector <- Vector`. Applying it to a ring of reference separations broadcasts over the beads, giving every bead's relative acceleration at every time for every polarization. Integrating twice from rest gives the displacements. The detector is much smaller than the wavelength, so the acceleration acts on each bead's unperturbed separation.


In [ ]:
# 1. The observer's tidal map for the whole packet:
response: Tidal = waves(t.wedge(Vector)).commutator(t)                 # [n_time, n_polarizations] Vector <- Vector

# 2. A ring of beads at radius 0.01 in the plane transverse to the wave, broadcast against the batch of
#    tidal maps. Each bead is x turned toward y by its angle; x and y square to minus one, which turns the sense:
angles = np.linspace(0, 2 * np.pi, 24, endpoint=False)
reference: Vector = (mv.xy * (angles / 2)).exp() >> x * 0.01           # [n_beads] Vector
acceleration: Vector = response[:, :, None](reference)                 # [n_time, n_polarizations, n_beads] Vector

# 3. Integrate twice from rest (numerical boundary):
displacement: Vector = integrate_acceleration(time, acceleration)      # [n_time, n_polarizations, n_beads] Vector

# Plot: the three polarizations at one shared instant, displacements magnified for display:
fig = render.draw_detector(time, reference, displacement, acceleration, amplification=4000)
plt.show()


## 7. The Same Wave as a Strain Map

In gauge theory gravity the wave is carried by a gauge field rather than a metric: a map on vectors that differs from the identity by a strain map, whose value on a rest separation is the physical separation. For the plus polarization the strain stretches along x and squeezes along y, and the cross pattern is the same map turned by 45 degrees, as the curvature was. Each bead's displacement is the strain map applied to its rest separation, with no integration. The curvature is the strain's second derivative along the wave: wedge the wave vector into that second derivative and weight by the overlap with the wave vector, and the result is the curvature as a map on pairs of vectors, the two edges of the area. It agrees with the dyad construction of section 1 at every time and polarization.

In general relativity the strain map reads as half the metric perturbation, $\tfrac{1}{2} h_{ij}$.

In [ ]:
# 1. Unit strain patterns on separations: stretch along x and squeeze along y, and the same turned by 45 degrees.
#    The strain acts on vectors, where I gives trivectors. The transverse plane xy also squares to minus one,
#    and turning each output a quarter turn turns the pattern by an eighth:
plus_strain: Strain = y * (y | Vector) - x * (x | Vector)                    # [] Vector <- Vector
cross_strain: Strain = mv.xy | plus_strain                                  # [] Vector <- Vector

# 2. The strain as a map on separations, batched over time and stacked over polarization:
#    plus + I * cross pairs the two patterns as the phasor pairs its parts: the vector part of the phasor
#    times it is the strain, for the scalar part alone, the pseudoscalar part alone, and the whole phasor:
analytic = plus_strain + I * cross_strain                                    # [] Vector + Trivector <- Vector
strain_map: Strain = 0.5 * stack((strain.select[0] * analytic, strain.select[4] * analytic, strain * analytic), axis=1).select[1]  # [n_time, n_polarizations] Vector <- Vector

# 3. Each bead's displacement is the strain map applied to its rest separation, and the integrated ring lands on it:
predicted: Vector = strain_map[:, :, None](reference)                        # [n_time, n_polarizations, n_beads] Vector
# Displacements are spacelike and square to minus their length squared:
error: Vector = displacement - predicted                                     # [n_time, n_polarizations, n_beads] Vector
mismatch: Scalar = (-error.norm_squared()).square_root()                     # [n_time, n_polarizations, n_beads] Scalar
peak: Scalar = (-predicted.norm_squared()).square_root()                     # [n_time, n_polarizations, n_beads] Scalar

# 4. Curvature from the strain's second derivative. Each factor carries its own open Vector,
#    k.wedge(second_strain) and (k | Vector), so their product has two slots, in the order they
#    appear: a map from the two edges of an area to a bivector, Bivector <- (Vector, Vector).
second_strain: Strain = 0.5 * stack((second.select[0] * analytic, second.select[4] * analytic, second * analytic), axis=1).select[1]  # [n_time, n_polarizations] Vector <- Vector
curvature_two_form = k.wedge(second_strain) * (k | Vector) - (k | Vector) * k.wedge(second_strain)   # [n_time, n_polarizations] Bivector <- (Vector, Vector)

# 5. Calling it with one edge binds the first slot and leaves the second open: Bivector <- Vector.
#    Compare with the dyad curvature applied to the wedge with that edge:
edge: Vector = mv.vector(np.random.default_rng(5).normal(size=4))            # [] Vector
from_strain = curvature_two_form(edge)                                  # [n_time, n_polarizations] Bivector <- Vector
from_dyads = waves(edge.wedge(Vector))                                       # [n_time, n_polarizations] Bivector <- Vector
agreement: Scalar = (from_strain - from_dyads).svdvals()                     # [n_time, n_polarizations, n_singular] Scalar

print("max |integrated - strain(reference)| :", mismatch.to_array().max())
print("peak displacement                    :", peak.to_array().max())
print("curvature from strain vs from dyads  :", agreement.to_array().max())
print(curvature_two_form)

## 8. Animation

Released from rest, the ring returns to rest once the packet has passed, to first order and to within the Gaussian's tails. In the circular case the elliptical pattern turns while each coloured bead traces a small loop about its rest position: a turning deformation, not a rigid rotation.


In [ ]:
frames = render.animate_detector(time, reference, displacement, acceleration, amplification=4000)
gif = save_animation(frames, "curvature", 50)
display(Image(filename=gif))

## 9. Summary

* **Curvature as a dyad sum**: `nx * (nx | Bivector)` with the area left open builds the Riemann map of a plane wave; opposite weights on the two null dyads make it vacuum.
* **Polarization by duality**: `-I * plus` is the cross pattern, the plus pattern turned by 45 degrees; for a null curvature a duality rotation turns the pattern by half its angle, and a wave packet's phase is `(I * -phase).exp()`.
* **Nilpotent but not zero**: the null outputs lie in the map's own kernel, so all six eigenvalues vanish. The indefinite metric permits this for a self-adjoint map.
* **Observer binding is composition, not conjugation**: `curvature(t.wedge(Vector)).commutator(t)` has eigenvalues plus and minus the amplitude although the curvature has none, and a boosted observer sees them scaled by the exponential of minus twice the rapidity.
* **Batches carry through**: time, polarization and observer rapidity are batch axes on the maps; beads broadcast against them, so the whole detector response is one expression.

* **The same wave as a strain map**: in gauge theory gravity the wave is a map on vectors; the strain applied to a bead's rest separation is its displacement, and the strain's second derivative wedged with the wave vector is the curvature, `k.wedge(second) * (k | Vector) - (k | Vector) * k.wedge(second)`.